# Download Billboard Audio and JAMS

This notebook downloads the Billboard dataset with `mirdata`, creates the export folder if it does not exist, and copies the downloaded audio and JAMS files into that folder.

In [11]:
from pathlib import Path
import shutil

import mirdata
import mirdata.datasets.billboard as billboard

# Shared root for all downloaded datasets.
DATASET_ROOT = Path("datasets")
# Billboard will be exported inside the shared datasets folder.
EXPORT_ROOT = DATASET_ROOT / "billboard_dataset"
RAW_DATA_HOME = EXPORT_ROOT / "_mirdata_raw"
AUDIO_EXPORT_DIR = EXPORT_ROOT / "audio"
JAMS_EXPORT_DIR = EXPORT_ROOT / "jams"

for folder in (DATASET_ROOT, EXPORT_ROOT, RAW_DATA_HOME, AUDIO_EXPORT_DIR, JAMS_EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Export root: {EXPORT_ROOT.resolve()}")
print(f"Audio export dir: {AUDIO_EXPORT_DIR.resolve()}")
print(f"JAMS export dir: {JAMS_EXPORT_DIR.resolve()}")


Dataset root: /home/oriolfreixa/Documents/repos/uni/MirInharmonicAugmentation/datasets
Export root: /home/oriolfreixa/Documents/repos/uni/MirInharmonicAugmentation/datasets/billboard_dataset
Audio export dir: /home/oriolfreixa/Documents/repos/uni/MirInharmonicAugmentation/datasets/billboard_dataset/audio
JAMS export dir: /home/oriolfreixa/Documents/repos/uni/MirInharmonicAugmentation/datasets/billboard_dataset/jams


In [12]:
def load_billboard_dataset(data_home: Path):
    """Initialize Billboard in a way that works across mirdata versions."""
    try:
        return billboard.Dataset(data_home=str(data_home))
    except TypeError:
        return mirdata.initialize("billboard", data_home=str(data_home))


dataset = load_billboard_dataset(RAW_DATA_HOME)
dataset.download(
    partial_download=[
        "metadata",
        "annotation_salami",
        "annotation_lab",
        "annotation_mirex13",
        "annotation_chordino",
    ]
)
print(f"Downloaded Billboard annotations into: {RAW_DATA_HOME.resolve()}")
print("Note: Billboard audio is not downloadable through mirdata; audio export only works if the audio files already exist locally under RAW_DATA_HOME/audio.")


Downloaded Billboard annotations into: /home/oriolfreixa/Documents/repos/uni/MirInharmonicAugmentation/datasets/billboard_dataset/_mirdata_raw
Note: Billboard audio is not downloadable through mirdata; audio export only works if the audio files already exist locally under RAW_DATA_HOME/audio.


In [13]:
import jams
import soundfile as sf


def first_existing_path(track, attribute_names):
    for attribute_name in attribute_names:
        candidate = getattr(track, attribute_name, None)
        if candidate:
            candidate_path = Path(candidate)
            if candidate_path.exists():
                return candidate_path
    return None


def export_track_audio(track, output_dir: Path):
    try:
        audio = track.audio
    except Exception:
        return False

    if audio is None:
        return False

    signal, sample_rate = audio
    output_path = output_dir / f"{track.track_id}.wav"
    sf.write(output_path, signal, sample_rate)
    return True


def export_track_jams(track, output_dir: Path):
    lab_path = first_existing_path(track, ["lab_full_path"])
    if lab_path is None:
        return False

    annotation = jams.Annotation(namespace="chord")
    with lab_path.open("r", encoding="utf-8") as fhandle:
        for line in fhandle:
            parts = line.strip().split(maxsplit=2)
            if len(parts) != 3:
                continue
            start, end, label = parts
            start_time = float(start)
            end_time = float(end)
            annotation.append(
                time=start_time,
                duration=max(0.0, end_time - start_time),
                value=label,
                confidence=1.0,
            )

    jam = jams.JAMS()
    jam.file_metadata.title = getattr(track, "title", track.track_id) or track.track_id
    jam.file_metadata.artist = getattr(track, "artist", "") or ""
    jam.file_metadata.identifiers = {"track_id": track.track_id, "dataset": "billboard"}
    if len(annotation) > 0:
        annotation.annotation_metadata = jams.AnnotationMetadata(data_source="McGill Billboard full.lab")
        last_obs = annotation[-1]
        jam.file_metadata.duration = float(last_obs.time + last_obs.duration)
        annotation.duration = jam.file_metadata.duration
    jam.annotations.append(annotation)
    jam.save(output_dir / f"{track.track_id}.jams")
    return True


audio_count = 0
jams_count = 0
missing_audio = []
missing_jams = []

for track_id in dataset.track_ids:
    track = dataset.track(track_id)

    if export_track_audio(track, AUDIO_EXPORT_DIR):
        audio_count += 1
    else:
        missing_audio.append(track_id)

    if export_track_jams(track, JAMS_EXPORT_DIR):
        jams_count += 1
    else:
        missing_jams.append(track_id)

print(f"Exported {audio_count} audio files to {AUDIO_EXPORT_DIR.resolve()}")
print(f"Exported {jams_count} JAMS files to {JAMS_EXPORT_DIR.resolve()}")

if missing_audio:
    print(f"Tracks without exported audio: {len(missing_audio)}")
if missing_jams:
    print(f"Tracks without exported JAMS: {len(missing_jams)}")


KeyError: -1